# 3. pipeline: timm + Lightning + TorchMetrics

`timm` даёт единый интерфейс к сотням vision-моделей и их pretrained-конфигурациям.

Lightning организует
train/validation loops, устройства, mixed precision, callbacks, checkpoints и logs.

TorchMetrics корректно
накапливает метрики по эпохе.

Сравним **head-only** и **full fine-tuning** одной MobileNetV3.


## 1. Установка и импорты


In [ ]:
import importlib.util, subprocess, sys
requirements = {"timm": "timm>=1.0,<2", "lightning": "lightning>=2.5,<3", "torchmetrics": "torchmetrics>=1.7,<2"}
missing = [spec for module, spec in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import EuroSAT
import timm
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from lightning.pytorch.loggers import CSVLogger
from torchmetrics import MetricCollection
from torchmetrics.classification import (MulticlassAccuracy, MulticlassF1Score,
                                          MulticlassPrecision, MulticlassRecall)

SEED = 42; FAST_MODE = True; NUM_CLASSES = 10
L.seed_everything(SEED, workers=True)
print("torch", torch.__version__, "| timm", timm.__version__, "| lightning", L.__version__)


## 2. DataModule

`LightningDataModule` группирует download, split, transforms и DataLoaders. Это удобно для повторения экспериментов:
обе модели гарантированно получают один split.


In [ ]:
class EuroSATDataModule(L.LightningDataModule):
    def __init__(self, root="data", batch_size=128, image_size=96, fast_mode=True):
        super().__init__(); self.root = Path(root); self.batch_size = batch_size
        self.image_size = image_size; self.fast_mode = fast_mode

    def prepare_data(self):
        EuroSAT(self.root, download=True)

    def setup(self, stage=None):
        probe = timm.create_model("mobilenetv3_small_100", pretrained=True)
        cfg = timm.data.resolve_data_config(probe.pretrained_cfg)
        cfg["input_size"] = (3, self.image_size, self.image_size)
        self.train_transform = timm.data.create_transform(**cfg, is_training=True,
                                                          hflip=0.5, vflip=0.5)
        self.eval_transform = timm.data.create_transform(**cfg, is_training=False)
        base = EuroSAT(self.root); targets = np.asarray(base.targets); self.class_names = base.classes
        train_source = EuroSAT(self.root, transform=self.train_transform)
        val_source = EuroSAT(self.root, transform=self.eval_transform)
        rng = np.random.default_rng(SEED); train_idx, val_idx = [], []
        for class_id in range(len(self.class_names)):
            idx = np.flatnonzero(targets == class_id); rng.shuffle(idx)
            if self.fast_mode: train_idx.extend(idx[:500]); val_idx.extend(idx[500:650])
            else:
                cut = int(0.8 * len(idx)); train_idx.extend(idx[:cut]); val_idx.extend(idx[cut:])
        self.train_set = Subset(train_source, train_idx); self.val_set = Subset(val_source, val_idx)

    def train_dataloader(self):
        return DataLoader(self.train_set, batch_size=self.batch_size, shuffle=True, num_workers=2,
                          persistent_workers=True, pin_memory=torch.cuda.is_available())

    def val_dataloader(self):
        return DataLoader(self.val_set, batch_size=self.batch_size, shuffle=False, num_workers=2,
                          persistent_workers=True, pin_memory=torch.cuda.is_available())

dm = EuroSATDataModule(fast_mode=FAST_MODE); dm.prepare_data(); dm.setup()
xb, yb = next(iter(dm.train_dataloader()))
print("batch:", tuple(xb.shape), tuple(yb.shape), "| classes:", dm.class_names)


## 3. LightningModule: модель, steps, метрики и optimizer

`timm.create_model(..., num_classes=10)` сам заменяет classifier head. Режим `head` замораживает backbone;
режим `full` обучает всю сеть с двумя LR. `MetricCollection` считает accuracy, macro precision/recall/F1 по эпохе.
Macro-усреднение даёт одинаковый вес каждому классу.


In [ ]:
class LitEuroSAT(L.LightningModule):
    def __init__(self, mode="head", model_name="mobilenetv3_small_100"):
        super().__init__(); self.save_hyperparameters()
        self.net = timm.create_model(model_name, pretrained=True, num_classes=NUM_CLASSES)
        if mode == "head":
            for p in self.net.parameters(): p.requires_grad = False
            for p in self.net.get_classifier().parameters(): p.requires_grad = True
        metrics = MetricCollection({
            "accuracy": MulticlassAccuracy(NUM_CLASSES, average="micro"),
            "precision_macro": MulticlassPrecision(NUM_CLASSES, average="macro"),
            "recall_macro": MulticlassRecall(NUM_CLASSES, average="macro"),
            "f1_macro": MulticlassF1Score(NUM_CLASSES, average="macro"),
        })
        self.train_metrics = metrics.clone(prefix="train_")
        self.val_metrics = metrics.clone(prefix="val_")
        self.loss_fn = nn.CrossEntropyLoss(label_smoothing=0.05)

    def forward(self, x): return self.net(x)

    def _shared_step(self, batch, stage):
        x, y = batch; logits = self(x); loss = self.loss_fn(logits, y)
        metrics = self.train_metrics if stage == "train" else self.val_metrics
        metrics.update(logits, y)
        self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True, prog_bar=True, batch_size=len(y))
        return loss

    def training_step(self, batch, batch_idx): return self._shared_step(batch, "train")
    def validation_step(self, batch, batch_idx): self._shared_step(batch, "val")

    def on_train_epoch_end(self):
        self.log_dict(self.train_metrics.compute(), prog_bar=False)
        self.train_metrics.reset()

    def on_validation_epoch_end(self):
        # Compute после всех batch: это global TP/FP/FN, а не среднее batch-метрик.
        self.log_dict(self.val_metrics.compute(), prog_bar=True)
        self.val_metrics.reset()

    def on_train_epoch_start(self):
        if self.hparams.mode == "head":
            for module in self.net.modules():
                if isinstance(module, nn.modules.batchnorm._BatchNorm): module.eval()

    def configure_optimizers(self):
        head = list(self.net.get_classifier().parameters()); head_ids = {id(p) for p in head}
        if self.hparams.mode == "head":
            groups = [{"params": head, "lr": 2e-3}]
        else:
            backbone = [p for p in self.net.parameters() if id(p) not in head_ids]
            groups = [{"params": backbone, "lr": 5e-5}, {"params": head, "lr": 5e-4}]
        optimizer = torch.optim.AdamW(groups, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.trainer.max_epochs)
        return {"optimizer": optimizer, "lr_scheduler": scheduler}


## 4. Два контролируемых эксперимента

`ModelCheckpoint` сохраняет лучший `val_f1_macro`, `EarlyStopping` прекращает бесполезный run, `CSVLogger`
создаёт таблицу метрик. `precision="16-mixed"` включается только на CUDA.


In [ ]:
experiments = {}
for mode in ["head", "full"]:
    model = LitEuroSAT(mode=mode)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    checkpoint = ModelCheckpoint(monitor="val_f1_macro", mode="max", save_top_k=1,
                                 filename=mode + "-{epoch:02d}-{val_f1_macro:.3f}")
    logger = CSVLogger("logs", name="eurosat_" + mode)
    trainer = L.Trainer(
        max_epochs=2 if FAST_MODE else 8, accelerator="auto", devices=1,
        precision="16-mixed" if torch.cuda.is_available() else "32-true",
        logger=logger, callbacks=[checkpoint, EarlyStopping(monitor="val_f1_macro", mode="max", patience=3)],
        deterministic=True, log_every_n_steps=10,
    )
    print(f"\n=== {mode}: trainable={trainable:,} ===")
    trainer.fit(model, datamodule=dm)
    validation = trainer.validate(model, datamodule=dm, ckpt_path="best", verbose=False)[0]
    experiments[mode] = {"metrics": validation, "log_dir": logger.log_dir,
                         "checkpoint": checkpoint.best_model_path, "trainable": trainable}


## 5. Анализ результатов и логов


In [ ]:
rows = []
for mode, result in experiments.items():
    row = {"mode": mode, "trainable": result["trainable"], **result["metrics"]}; rows.append(row)
comparison = pd.DataFrame(rows).set_index("mode")
display(comparison)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for mode, result in experiments.items():
    log = pd.read_csv(Path(result["log_dir"]) / "metrics.csv")
    val = log.dropna(subset=["val_loss"])
    axes[0].plot(val["epoch"], val["val_loss"], "o-", label=mode)
    axes[1].plot(val["epoch"], val["val_f1_macro"], "o-", label=mode)
axes[0].set(title="Validation loss", xlabel="epoch"); axes[1].set(title="Macro-F1", xlabel="epoch", ylim=(0, 1))
for ax in axes: ax.grid(); ax.legend()
plt.show()
print("Best checkpoints:", *(f"{mode}: {r['checkpoint']}" for mode, r in experiments.items()), sep="\n")


## Практика и квиз

1. Добавьте третий mode, размораживающий только последний stage MobileNetV3.
2. Измените monitor с accuracy на macro-F1 и объясните, когда это меняет выбранный checkpoint.
3. Запустите `fast_dev_run=True`, затем намеренно сломайте число классов и прочитайте traceback.
4. Сравните `mobilenetv3_small_100` и `resnet18` по параметрам, времени и F1.

Вопросы: Почему нельзя сравнивать runs с разными splits?
Что делает `on_epoch=True`? Почему full model получает меньший backbone LR? Что сохраняет checkpoint?
Какая метрика сильнее реагирует на слабый редкий класс: micro accuracy или macro-F1?


## Источники

- [EuroSAT paper](https://arxiv.org/abs/1709.00029)
- [timm Quickstart](https://huggingface.co/docs/timm/quickstart)
- [Lightning Trainer](https://lightning.ai/docs/pytorch/stable/common/trainer.html)
- [TorchMetrics classification overview](https://lightning.ai/docs/torchmetrics/stable/pages/overview.html)
- [TorchVision EuroSAT](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.EuroSAT.html)
